# Phase 3: Energy Efficiency & Process Optimization — Panificadora Chask

**Engineering narrative**: This notebook analyses the energy and process outcomes
of the 2021 plant modernization. All heavy computation is delegated to the
`chask` package; the notebook is for narrative and visualization only.

> **Data disclosure**: All monthly statistics use `monthly_reconstructed.csv`
> — a documented reconstruction calibrated to the engineering report.
> The daily synthetic dataset is used **only** for the load profile section,
> clearly labeled as demonstration only.

## Setup

In [ ]:
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings('ignore')
from chask.pipeline.ingest import load_raw
from chask.pipeline.validate import validate
from chask.pipeline.transform import to_analytics
from chask.config import ANALYTICS_DIR
df = to_analytics(validate(load_raw()))
pre = df[df['period'] == 'pre']
post = df[df['period'] == 'post']
ss  = df[df['fecha'] >= '2021-12-31']
print(f'Dataset: {len(df)} months | pre={len(pre)} post={len(post)} SS={len(ss)}')

## 1. Energy KPIs

**Why do old motors consume more?**
Older induction motors (>5 years service) suffer degraded windings, bearing wear,
and insulation deterioration. These defects increase resistive losses (I²R) in the
stator and rotor, converting electrical energy to heat. An IE1 motor at 75–82%
efficiency wastes 18–25% of every kWh. An IE3 motor at 90–93% wastes only 7–10%.

In [ ]:
from chask.energy.kpis import annualized_savings, energy_intensity_rolling, monthly_savings
sav = annualized_savings(df)
print('=== Annualized savings (SS vs pre baseline) ===')
for k, v in sav.items():
    print(f'  {k:30s} {v}')

In [ ]:
rolled = energy_intensity_rolling(df)
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df['fecha'], df['intensity_kwh_kg'], 'o', alpha=0.5, ms=5,
        label='Monthly', color='#607D8B')
ax.plot(rolled['fecha'], rolled['intensity_rolling'], '-', lw=2,
        color='#E91E63', label='3-mo rolling avg')
ax.axvline(pd.Timestamp('2021-08-31'), ls='--', color='#9C27B0', label='Intervention cutoff')
ax.axvline(pd.Timestamp('2021-11-30'), ls=':', color='#009688', label='SS start (Dec 2021)')
ax.set(xlabel='Month', ylabel='kWh/kg',
       title='Energy intensity — monthly and 3-month rolling average')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 2. Motor Replacement Savings

**What is IE3?** IEC 60034-30-1 classifies motor efficiency into tiers:
IE1 = standard, IE2 = high, IE3 = premium, IE4 = super-premium.
IE3 motors in the 2–15 kW range achieve 88.5–93.0% efficiency vs.
75–82% for degraded IE1 motors. The savings formula:

    savings_kWh = P_shaft × hours × (1/η_old − 1/η_new)

In [ ]:
from chask.energy.motors import motor_savings_detail, reconciliation, total_motor_savings_kwh_mo
detail = motor_savings_detail()
print(detail.to_string(index=False))
print(f'\nFleet total theoretical savings: {total_motor_savings_kwh_mo():.0f} kWh/mo')

In [ ]:
observed_mo = sav['baseline_kwh_mo'] - sav['ss_mean_kwh_mo']
rec = reconciliation(observed_mo)
print('\n=== Reconciliation: theoretical vs. observed ===')
for k, v in rec.items():
    print(f'  {k:35s} {v}')

## 3. Process Optimization: Throughput and Reliability

**Why does MTBF improve with preventive maintenance?**
Reactive (breakdown) maintenance allows minor faults to escalate into catastrophic
failures. A preventive programme intercepts wear early (bearings, belts, lubrication),
keeping components within design tolerances. The transition period (Sep–Nov 2021)
shows elevated failures during commissioning — a normal startup pattern.

In [ ]:
from chask.process.throughput import throughput_summary
from chask.process.reliability import failure_trends, reliability_summary
print('=== Throughput by period ===')
print(throughput_summary(df).to_string())
print('\n=== Failure trends ===')
print(failure_trends(df).to_string())
print('\n=== Reliability summary ===')
for k, v in reliability_summary(df).items():
    print(f'  {k:40s} {v}')

## 4. ROI Analysis — Three Scenarios

The USD 85,000 investment covers: IE3 motors, PLC automation, rewiring, and
preventive maintenance programme setup.

Scenarios use documented assumptions only (no magic numbers):
- **Conservative**: energy savings + downtime reduction
- **Base**: + incremental margin from observed production growth
- **Optimistic**: + progressive +50% capacity utilization over 3 years

In [ ]:
from chask.energy.roi import compute_roi_scenarios, payback_curves
roi = compute_roi_scenarios(df)
print(roi.to_string(index=False))

In [ ]:
curves = payback_curves(df)
palette = {'Conservative': '#F44336', 'Base': '#FF9800', 'Optimistic': '#4CAF50'}
fig, ax = plt.subplots(figsize=(11, 5))
for s in ['Conservative', 'Base', 'Optimistic']:
    ax.plot(curves['year'], curves[s], '-o', lw=2, color=palette[s], label=s)
ax.axhline(0, color='black', lw=1, ls='--')
ax.set(xlabel='Year', ylabel='Cumulative discounted CF (USD)',
       title='ROI payback curves — USD 85,000 investment @ 10% discount rate')
ax.legend(); ax.grid(alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout(); plt.show()
print('Conclusion: energy savings alone (conservative) do not cover the investment.')
print('The full case requires capacity utilization growth (optimistic).')

## 5. Load Profile (Synthetic Daily — Demonstration Only)

> ⚠️ **WARNING**: The following uses **`daily_synthetic.csv`** — model-generated data,
> NOT real measurements. Do NOT use for financial reporting, regulatory submissions,
> or statistical inference.

In [ ]:
from chask.energy.load_profile import weekly_load_profile, peak_day_of_week
daily_df = pd.read_csv(ANALYTICS_DIR / 'daily_synthetic.csv', parse_dates=['fecha'])
profile = weekly_load_profile(daily_df)
peak = peak_day_of_week(daily_df)
print(f'Peak day: {peak["day_label"]} ({peak["mean_kwh"]:.1f} kWh/day)')
print(profile[['day_label', 'mean_kwh', 'std_kwh']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(profile['day_label'], profile['mean_kwh'], color='#2196F3', alpha=0.7)
ax.errorbar(profile['day_label'], profile['mean_kwh'], yerr=profile['std_kwh'],
            fmt='none', color='black', capsize=4)
ax.set(xlabel='Day of week', ylabel='Mean daily energy (kWh)',
       title='Weekly load profile — SYNTHETIC DATA ONLY (demonstration)')
ax.grid(alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

## Summary

| Metric | Pre | Steady-state | Change |
|---|---|---|---|
| Energy (kWh/mo) | 51,827 | 40,062 | −22.7% |
| MTBF (h) | 88.7 | 316.1 | +256% |
| Productivity (kg/op-hr) | 19.7 | 22.4 | +13.3% |
| ROI NPV 5yr (conservative) | — | — | −$42,430 |
| ROI NPV 5yr (optimistic) | — | — | +$69,618 |

**Motor savings explain ~61% of total energy reduction.** Non-motor improvements
(connections, phantom load, automation) account for the remaining ~39%.
The investment is justified only when capacity utilization growth is included.